# 03 - Embeddings

RAG needs to find the documents most relevant to a question. But how does a
computer measure that "the leave policy" is relevant to "how many vacation
days do I get" when the two share almost no words? The answer is embeddings.

**What you will learn**

- What an embedding is
- How to create embeddings with the API
- How cosine similarity turns two embeddings into a relevance score
- How to find the most relevant sentence for a question

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"
print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


## What is an embedding?

An embedding is a list of numbers that represents the meaning of a piece of
text. An embedding model reads text and outputs a fixed-length vector,
typically with hundreds or thousands of numbers.

The key property: **texts with similar meaning get vectors that are close
together.** "vacation days" and "annual leave" end up near each other, even
though the words differ. That is exactly the relevance measure RAG needs.

Let us embed one sentence and look at the result.

In [2]:
response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input="Employees receive 24 days of paid annual leave.",
)
vector = response.data[0].embedding

print("Vector length:", len(vector))
print("First 5 numbers:", [round(x, 4) for x in vector[:5]])

Vector length: 1536
First 5 numbers: [-0.0188, 0.0261, 0.0514, 0.0406, -0.0055]


1536 numbers. Each number means nothing on its own. Only comparisons
between whole vectors are meaningful.

## Comparing vectors: cosine similarity

The standard way to compare two embeddings is cosine similarity: the cosine
of the angle between the two vectors.

- **1.0** means identical direction (same meaning)
- **around 0** means unrelated
- texts about similar topics typically land between 0.3 and 0.9

The formula is: dot product of the vectors, divided by the product of their
lengths. In plain Python:

In [3]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    length_a = sum(x * x for x in a) ** 0.5
    length_b = sum(x * x for x in b) ** 0.5
    return dot / (length_a * length_b)

## Seeing similarity in action

We embed a few sentences: two about employee leave, two about robots, one
about cooking. The API accepts a list, so one call embeds them all.

In [4]:
sentences = [
    "Employees receive 24 days of paid annual leave per year.",
    "How much vacation time do staff members get?",
    "The Carrier X2 robot can carry a payload of 25 kilograms.",
    "Our delivery robots navigate hospitals using lidar sensors.",
    "Add two spoons of sugar and stir the batter well.",
]

response = client.embeddings.create(model=EMBEDDING_MODEL, input=sentences)
vectors = [item.embedding for item in response.data]
print(f"Embedded {len(vectors)} sentences.")

Embedded 5 sentences.


In [5]:
print("Similarity between every pair of sentences:\n")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        score = cosine_similarity(vectors[i], vectors[j])
        print(f"  {score:.3f}  [{i}] vs [{j}]")

print()
for i, s in enumerate(sentences):
    print(f"[{i}] {s}")

Similarity between every pair of sentences:

  0.548  [0] vs [1]
  0.117  [0] vs [2]
  0.089  [0] vs [3]
  0.015  [0] vs [4]
  0.049  [1] vs [2]
  0.007  [1] vs [3]
  0.019  [1] vs [4]
  0.273  [2] vs [3]
  0.096  [2] vs [4]
  -0.015  [3] vs [4]

[0] Employees receive 24 days of paid annual leave per year.
[1] How much vacation time do staff members get?
[2] The Carrier X2 robot can carry a payload of 25 kilograms.
[3] Our delivery robots navigate hospitals using lidar sensors.
[4] Add two spoons of sugar and stir the batter well.


Look at the scores:

- Sentence 0 (annual leave) and sentence 1 (vacation time) score highest,
  despite sharing almost no words. The embedding model understood they mean
  the same thing.
- The two robot sentences (2 and 3) also score high with each other.
- The cooking sentence (4) scores low against everything.

## Using this for search

Now the core move of RAG retrieval: embed a question, compare it against
every stored text, and take the best match.

In [6]:
question = "How many days off do employees get?"

q_vector = client.embeddings.create(model=EMBEDDING_MODEL, input=question).data[0].embedding

scores = [cosine_similarity(q_vector, v) for v in vectors]

print(f"Question: {question}\n")
for score, sentence in sorted(zip(scores, sentences), reverse=True):
    print(f"  {score:.3f}  {sentence}")

Question: How many days off do employees get?

  0.600  How much vacation time do staff members get?
  0.559  Employees receive 24 days of paid annual leave per year.
  0.072  Our delivery robots navigate hospitals using lidar sensors.
  0.022  The Carrier X2 robot can carry a payload of 25 kilograms.
  -0.014  Add two spoons of sugar and stir the batter well.


The leave sentences rise to the top. This is semantic search: search by
meaning instead of by matching keywords.

Comparing a question against five sentences is easy. Comparing it against a
million chunks needs a purpose-built tool: a vector database, coming in
notebook 05. But first, notebook 04 answers: what exactly should we embed and
store? Whole documents are too big. We need to cut them into chunks.

## Ollama alternative

Ollama serves embedding models through the same API. Uncomment to use it:

In [7]:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# EMBEDDING_MODEL = "nomic-embed-text"
# Then re-run the cells above. Note: vectors from different embedding
# models are never comparable with each other.

## Exercise

1. Add a sentence about the weather to the sentences list and re-run the
   similarity cells. Which existing sentence is it closest to?
2. Change the question to "What sensors do the robots use?" and re-run the
   search cell. Does the right sentence win?

In [8]:
# Try the exercise here
